## Imports

In [4]:
import json
import re
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

ModuleNotFoundError: No module named 'sklearn'

## Part 1: Multiple Choice Evaluation

In [3]:
MC_DIR = Path("../data/results/multichoice")

def extract_option_letter(val) -> str:
    if not val:
        return "INVALID"
    if isinstance(val, dict):
        return str(val.get("selected_option", "INVALID")).strip().upper()
    val_str = str(val).strip()

    # Try parsing embedded JSON
    try:
        parsed = json.loads(val_str)
        if isinstance(parsed, dict) and "selected_option" in parsed:
            return str(parsed["selected_option"]).strip().upper()
    except Exception:
        pass

    # Regex fallback for option letters
    match = re.search(r'["\']?selected_option["\']?\s*:\s*["\']?([A-D])["\']?', val_str, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    direct_match = re.search(r'\b([A-D])\b', val_str)
    if direct_match:
        return direct_match.group(1).upper()

    return "INVALID"

mc_results = {}

for p in sorted(MC_DIR.glob("*.json")):
    model_name = p.stem
    with p.open("r", encoding="utf-8") as f:
        records = json.load(f)

    y_true = []
    y_pred = []

    for r in records:
        gt = str(r.get("correct_label", "")).strip().upper()
        raw_pred = r.get("predicted_label") if r.get("predicted_label") is not None else r.get("raw_model_response")
        pred = extract_option_letter(raw_pred)

        y_true.append(gt)
        y_pred.append(pred)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=["A", "B", "C", "D"], zero_division=0)

    # Distribution of selections
    pred_counts = pd.Series(y_pred).value_counts().to_dict()

    mc_results[model_name] = {
        "Accuracy (%)": round(acc * 100, 2),
        "Macro F1": round(macro_f1, 4),
        "Valid Formats": sum(1 for x in y_pred if x in ["A", "B", "C", "D"]),
        "Total": len(y_true),
        "A": pred_counts.get("A", 0),
        "B": pred_counts.get("B", 0),
        "C": pred_counts.get("C", 0),
        "D": pred_counts.get("D", 0),
        "Invalid": pred_counts.get("INVALID", 0)
    }

df_mc = pd.DataFrame.from_dict(mc_results, orient="index").sort_values(by="Accuracy (%)", ascending=False)
df_mc

NameError: name 'accuracy_score' is not defined